In [1]:
import sys
print(sys.executable)

/opt/miniconda3/envs/har/bin/python


In [2]:
import pandas as pd
import sklearn
print("ok")

ok


In [3]:
import os
import json
import time
import sqlite3
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd

In [4]:
RANDOM_STATE = 42

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data" / "UCI HAR Dataset"
RESULTS_DIR = PROJECT_ROOT / "results"
DB_PATH = RESULTS_DIR / "experiments.db"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_DIR exists:", DATA_DIR.exists())
print("RESULTS_DIR exists:", RESULTS_DIR.exists())
print("DB_PATH:", DB_PATH)

PROJECT_ROOT: /Users/arshia/Projects/signal-activity-classification
DATA_DIR exists: True
RESULTS_DIR exists: True
DB_PATH: /Users/arshia/Projects/signal-activity-classification/results/experiments.db


In [5]:
def load_ucihar(data_dir: Path):
    X_train = pd.read_csv(data_dir / "train" / "X_train.txt", sep=r"\s+", header=None)
    X_test = pd.read_csv(data_dir / "test" / "X_test.txt", sep=r"\s+", header=None)

    y_train = pd.read_csv(data_dir / "train" / "y_train.txt", sep=r"\s+", header=None).iloc[:, 0]
    y_test = pd.read_csv(data_dir / "test" / "y_test.txt", sep=r"\s+", header=None).iloc[:, 0]

    features = pd.read_csv(
        data_dir / "features.txt",
        sep=r"\s+",
        header=None,
        names=["index", "feature_name"]
    )

    activity_labels = pd.read_csv(
        data_dir / "activity_labels.txt",
        sep=r"\s+",
        header=None,
        names=["label_id", "activity_name"]
    )

    label_map = dict(zip(activity_labels["label_id"], activity_labels["activity_name"]))

    X_train.columns = features["feature_name"].values
    X_test.columns = features["feature_name"].values

    y_train = y_train.map(label_map)
    y_test = y_test.map(label_map)

    return X_train, X_test, y_train, y_test, features, activity_labels

In [7]:
X_train, X_test, y_train, y_test, features_df, activity_labels_df = load_ucihar(DATA_DIR)

In [8]:
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("Number of features:", X_train.shape[1])
print("Unique labels:", sorted(y_train.unique()))
print("Missing values in X_train:", X_train.isna().sum().sum())

X_train shape: (7352, 561)
X_test shape: (2947, 561)
Number of features: 561
Unique labels: ['LAYING', 'SITTING', 'STANDING', 'WALKING', 'WALKING_DOWNSTAIRS', 'WALKING_UPSTAIRS']
Missing values in X_train: 0


In [9]:
display(X_train.head(2))
display(y_train.head(10))
display(activity_labels_df)

,tBodyAcc-mean()-X,tBodyAcc-mean()-Y,tBodyAcc-mean()-Z,tBodyAcc-std()-X,tBodyAcc-std()-Y,tBodyAcc-std()-Z,tBodyAcc-mad()-X,tBodyAcc-mad()-Y,tBodyAcc-mad()-Z,tBodyAcc-max()-X,...,fBodyBodyGyroJerkMag-meanFreq(),fBodyBodyGyroJerkMag-skewness(),fBodyBodyGyroJerkMag-kurtosis(),"angle(tBodyAccMean,gravity)","angle(tBodyAccJerkMean),gravityMean)","angle(tBodyGyroMean,gravityMean)","angle(tBodyGyroJerkMean,gravityMean)","angle(X,gravityMean)","angle(Y,gravityMean)","angle(Z,gravityMean)"
0,0.288585,-0.020294,-0.132905,-0.995279,-0.983111,-0.913526,-0.995112,-0.983185,-0.923527,-0.934724,...,-0.074323,-0.298676,-0.710304,-0.112754,0.030400,-0.464761,-0.018446,-0.841247,0.179941,-0.058627
1,0.278419,-0.016411,-0.123520,-0.998245,-0.975300,-0.960322,-0.998807,-0.974914,-0.957686,-0.943068,...,0.158075,-0.595051,-0.861499,0.053477,-0.007435,-0.732626,0.703511,-0.844788,0.180289,-0.054317


0    STANDING
1    STANDING
2    STANDING
3    STANDING
4    STANDING
5    STANDING
6    STANDING
7    STANDING
8    STANDING
9    STANDING
Name: 0, dtype: str

,label_id,activity_name
0,1,WALKING
1,2,WALKING_UPSTAIRS
2,3,WALKING_DOWNSTAIRS
3,4,SITTING
4,5,STANDING
5,6,LAYING


In [10]:
def init_db(db_path: Path):
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()

    cursor.execute("""
    CREATE TABLE IF NOT EXISTS experiments (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        run_name TEXT,
        model_name TEXT NOT NULL,
        params_json TEXT NOT NULL,
        random_state INTEGER,
        train_samples INTEGER,
        test_samples INTEGER,
        n_features INTEGER,
        accuracy REAL,
        macro_f1 REAL,
        train_time_sec REAL,
        inference_time_sec REAL,
        model_memory_bytes INTEGER,
        notes TEXT,
        created_at TEXT NOT NULL
    )
    """)

    conn.commit()
    conn.close()

In [11]:
init_db(DB_PATH)

In [12]:
print("DB_PATH:", DB_PATH)
print("DB exists:", DB_PATH.exists())

DB_PATH: /Users/arshia/Projects/signal-activity-classification/results/experiments.db
DB exists: True


In [13]:
conn = sqlite3.connect(DB_PATH)

tables_df = pd.read_sql_query("""
SELECT name
FROM sqlite_master
WHERE type='table'
ORDER BY name
""", conn)

conn.close()

tables_df

,name
0,experiments
1,sqlite_sequence


In [14]:
def log_experiment(
    db_path: Path,
    run_name: str,
    model_name: str,
    params: dict,
    random_state: int,
    train_samples: int,
    test_samples: int,
    n_features: int,
    accuracy: float,
    macro_f1: float,
    train_time_sec: float,
    inference_time_sec: float,
    model_memory_bytes: int,
    notes: str = ""
):
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()

    cursor.execute("""
    INSERT INTO experiments (
        run_name, model_name, params_json, random_state,
        train_samples, test_samples, n_features,
        accuracy, macro_f1, train_time_sec, inference_time_sec,
        model_memory_bytes, notes, created_at
    )
    VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
    """, (
        run_name,
        model_name,
        json.dumps(params, ensure_ascii=False),
        random_state,
        train_samples,
        test_samples,
        n_features,
        accuracy,
        macro_f1,
        train_time_sec,
        inference_time_sec,
        model_memory_bytes,
        notes,
        datetime.now().isoformat(timespec="seconds")
    ))

    conn.commit()
    conn.close()

In [15]:
log_experiment(
    db_path=DB_PATH,
    run_name="test_insert_001",
    model_name="DummyModel",
    params={"test_param": 123},
    random_state=RANDOM_STATE,
    train_samples=len(X_train),
    test_samples=len(X_test),
    n_features=X_train.shape[1],
    accuracy=0.0,
    macro_f1=0.0,
    train_time_sec=0.0,
    inference_time_sec=0.0,
    model_memory_bytes=0,
    notes="Test row to verify DB insert works"
)

In [16]:
conn = sqlite3.connect(DB_PATH)

latest_runs_df = pd.read_sql_query("""
SELECT
    id,
    run_name,
    model_name,
    accuracy,
    macro_f1,
    created_at
FROM experiments
ORDER BY id DESC
LIMIT 5
""", conn)

conn.close()

latest_runs_df

,id,run_name,model_name,accuracy,macro_f1,created_at
0,1,test_insert_001,DummyModel,0.0,0.0,2026-03-07T19:52:43


In [17]:
import pickle

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score
from sklearn.base import clone

In [18]:
def estimate_model_size_bytes(model):
    return len(pickle.dumps(model))

In [19]:
def train_and_evaluate(model, X_train, y_train, X_test, y_test):
    model = clone(model)

    start_train = time.perf_counter()
    model.fit(X_train, y_train)
    end_train = time.perf_counter()

    start_infer = time.perf_counter()
    y_pred = model.predict(X_test)
    end_infer = time.perf_counter()

    acc = accuracy_score(y_test, y_pred)
    macro_f1 = f1_score(y_test, y_pred, average="macro")

    result = {
        "model": model,
        "accuracy": acc,
        "macro_f1": macro_f1,
        "train_time_sec": end_train - start_train,
        "inference_time_sec": end_infer - start_infer,
        "model_memory_bytes": estimate_model_size_bytes(model),
        "y_pred": y_pred
    }

    return result

In [21]:
logreg = LogisticRegression(
    max_iter=1000,
    solver="lbfgs",
    random_state=RANDOM_STATE
)

In [22]:
logreg_result = train_and_evaluate(logreg, X_train, y_train, X_test, y_test)

In [23]:
logreg_result.keys()

dict_keys(['model', 'accuracy', 'macro_f1', 'train_time_sec', 'inference_time_sec', 'model_memory_bytes', 'y_pred'])

In [24]:
print("LogReg Accuracy:", round(logreg_result["accuracy"], 4))
print("LogReg Macro F1:", round(logreg_result["macro_f1"], 4))
print("Train Time (s):", round(logreg_result["train_time_sec"], 4))
print("Inference Time (s):", round(logreg_result["inference_time_sec"], 6))
print("Model Size (KB):", round(logreg_result["model_memory_bytes"] / 1024, 2))

LogReg Accuracy: 0.9606
LogReg Macro F1: 0.9606
Train Time (s): 1.9263
Inference Time (s): 0.015138
Model Size (KB): 39.34


In [25]:
log_experiment(
    db_path=DB_PATH,
    run_name="week2_logreg_baseline",
    model_name="LogisticRegression",
    params=logreg.get_params(),
    random_state=RANDOM_STATE,
    train_samples=len(X_train),
    test_samples=len(X_test),
    n_features=X_train.shape[1],
    accuracy=logreg_result["accuracy"],
    macro_f1=logreg_result["macro_f1"],
    train_time_sec=logreg_result["train_time_sec"],
    inference_time_sec=logreg_result["inference_time_sec"],
    model_memory_bytes=logreg_result["model_memory_bytes"],
    notes="Week 2 reproducible pipeline baseline"
)

In [26]:
conn = sqlite3.connect(DB_PATH)

runs_df = pd.read_sql_query("""
SELECT
    id,
    run_name,
    model_name,
    accuracy,
    macro_f1,
    train_time_sec,
    inference_time_sec,
    ROUND(model_memory_bytes / 1024.0, 2) AS model_size_kb,
    created_at
FROM experiments
ORDER BY id DESC
LIMIT 10
""", conn)

conn.close()

runs_df

,id,run_name,model_name,accuracy,macro_f1,train_time_sec,inference_time_sec,model_size_kb,created_at
0,2,week2_logreg_baseline,LogisticRegression,0.960638,0.960557,1.926265,0.015138,39.34,2026-03-11T00:23:14
1,1,test_insert_001,DummyModel,0.000000,0.000000,0.000000,0.000000,0.00,2026-03-07T19:52:43


In [27]:
rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=None,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

In [28]:
rf_result = train_and_evaluate(rf, X_train, y_train, X_test, y_test)

In [29]:
print("RF Accuracy:", round(rf_result["accuracy"], 4))
print("RF Macro F1:", round(rf_result["macro_f1"], 4))
print("RF Train Time (s):", round(rf_result["train_time_sec"], 4))
print("RF Inference Time (s):", round(rf_result["inference_time_sec"], 6))
print("RF Model Size (KB):", round(rf_result["model_memory_bytes"] / 1024, 2))

RF Accuracy: 0.9291
RF Macro F1: 0.9275
RF Train Time (s): 3.0573
RF Inference Time (s): 0.036445
RF Model Size (KB): 11694.07


In [30]:
log_experiment(
    db_path=DB_PATH,
    run_name="week2_rf_baseline",
    model_name="RandomForestClassifier",
    params=rf.get_params(),
    random_state=RANDOM_STATE,
    train_samples=len(X_train),
    test_samples=len(X_test),
    n_features=X_train.shape[1],
    accuracy=rf_result["accuracy"],
    macro_f1=rf_result["macro_f1"],
    train_time_sec=rf_result["train_time_sec"],
    inference_time_sec=rf_result["inference_time_sec"],
    model_memory_bytes=rf_result["model_memory_bytes"],
    notes="Week 2 architecture comparison baseline"
)

In [31]:
conn = sqlite3.connect(DB_PATH)

comparison_df = pd.read_sql_query("""
SELECT
    run_name,
    model_name,
    accuracy,
    macro_f1,
    train_time_sec,
    inference_time_sec,
    ROUND(model_memory_bytes / 1024.0, 2) AS model_size_kb
FROM experiments
WHERE run_name IN ('week2_logreg_baseline', 'week2_rf_baseline')
ORDER BY macro_f1 DESC, accuracy DESC
""", conn)

conn.close()

comparison_df

,run_name,model_name,accuracy,macro_f1,train_time_sec,inference_time_sec,model_size_kb
0,week2_logreg_baseline,LogisticRegression,0.960638,0.960557,1.926265,0.015138,39.34
1,week2_rf_baseline,RandomForestClassifier,0.929080,0.927509,3.057291,0.036445,11694.07


In [32]:
conn = sqlite3.connect(DB_PATH)

best_model_df = pd.read_sql_query("""
SELECT
    run_name,
    model_name,
    accuracy,
    macro_f1,
    train_time_sec,
    inference_time_sec,
    ROUND(model_memory_bytes / 1024.0, 2) AS model_size_kb
FROM experiments
WHERE run_name != 'test_insert_001'
ORDER BY macro_f1 DESC, accuracy DESC
LIMIT 5
""", conn)

conn.close()

best_model_df

,run_name,model_name,accuracy,macro_f1,train_time_sec,inference_time_sec,model_size_kb
0,week2_logreg_baseline,LogisticRegression,0.960638,0.960557,1.926265,0.015138,39.34
1,week2_rf_baseline,RandomForestClassifier,0.929080,0.927509,3.057291,0.036445,11694.07


In [33]:
rf_small = RandomForestClassifier(
    n_estimators=50,
    max_depth=10,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

In [34]:
rf_small_result = train_and_evaluate(rf_small, X_train, y_train, X_test, y_test)

In [35]:
print("RF Small Accuracy:", round(rf_small_result["accuracy"], 4))
print("RF Small Macro F1:", round(rf_small_result["macro_f1"], 4))
print("RF Small Train Time (s):", round(rf_small_result["train_time_sec"], 4))
print("RF Small Inference Time (s):", round(rf_small_result["inference_time_sec"], 6))
print("RF Small Model Size (KB):", round(rf_small_result["model_memory_bytes"] / 1024, 2))

RF Small Accuracy: 0.9192
RF Small Macro F1: 0.917
RF Small Train Time (s): 0.5789
RF Small Inference Time (s): 0.020777
RF Small Model Size (KB): 2040.85


In [36]:
log_experiment(
    db_path=DB_PATH,
    run_name="week2_rf_small",
    model_name="RandomForestClassifier",
    params=rf_small.get_params(),
    random_state=RANDOM_STATE,
    train_samples=len(X_train),
    test_samples=len(X_test),
    n_features=X_train.shape[1],
    accuracy=rf_small_result["accuracy"],
    macro_f1=rf_small_result["macro_f1"],
    train_time_sec=rf_small_result["train_time_sec"],
    inference_time_sec=rf_small_result["inference_time_sec"],
    model_memory_bytes=rf_small_result["model_memory_bytes"],
    notes="Smaller RF for parameter trade-off comparison"
)

In [37]:
conn = sqlite3.connect(DB_PATH)

final_comparison_df = pd.read_sql_query("""
SELECT
    run_name,
    model_name,
    accuracy,
    macro_f1,
    train_time_sec,
    inference_time_sec,
    ROUND(model_memory_bytes / 1024.0, 2) AS model_size_kb
FROM experiments
WHERE run_name IN (
    'week2_logreg_baseline',
    'week2_rf_baseline',
    'week2_rf_small'
)
ORDER BY macro_f1 DESC, accuracy DESC
""", conn)

conn.close()

final_comparison_df

,run_name,model_name,accuracy,macro_f1,train_time_sec,inference_time_sec,model_size_kb
0,week2_logreg_baseline,LogisticRegression,0.960638,0.960557,1.926265,0.015138,39.34
1,week2_rf_baseline,RandomForestClassifier,0.929080,0.927509,3.057291,0.036445,11694.07
2,week2_rf_small,RandomForestClassifier,0.919240,0.917022,0.578888,0.020777,2040.85


In [38]:
conn = sqlite3.connect(DB_PATH)

best_rf_df = pd.read_sql_query("""
SELECT
    run_name,
    model_name,
    accuracy,
    macro_f1,
    train_time_sec,
    inference_time_sec,
    ROUND(model_memory_bytes / 1024.0, 2) AS model_size_kb
FROM experiments
WHERE model_name = 'RandomForestClassifier'
ORDER BY macro_f1 DESC, accuracy DESC
LIMIT 5
""", conn)

conn.close()

best_rf_df

,run_name,model_name,accuracy,macro_f1,train_time_sec,inference_time_sec,model_size_kb
0,week2_rf_baseline,RandomForestClassifier,0.92908,0.927509,3.057291,0.036445,11694.07
1,week2_rf_small,RandomForestClassifier,0.91924,0.917022,0.578888,0.020777,2040.85


## Week 2 Summary

In week 2, I built a reproducible experimentation pipeline for the UCI HAR project with:
- clean data loading
- model training and evaluation
- SQLite-based experiment tracking
- benchmark comparison using training time, inference time, and estimated model size

Models compared:
- LogisticRegression
- RandomForestClassifier (baseline)
- RandomForestClassifier (smaller configuration)

Main findings:
- LogisticRegression achieved the best overall trade-off.
- RandomForest baseline had better accuracy than rf_small.
- rf_small was faster and smaller than the baseline RF.